In [1]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import math
from tqdm import tqdm
import os

In [2]:
rave_sim_dir = Path('/mnt/d/rave-sim-main/rave-sim-main')
simulations_dir = Path('/mnt/d/rave-sim-main/rave-sim-main/output')
scratch_dir = simulations_dir

In [3]:
sys.path.insert(0, str(rave_sim_dir / "big-wave"))
import multisim
import config
import util

In [4]:
# ===== 扫参：扫描 z_detector 位置，验证 2D 传播的 1/r² 强度衰减 =====

w_width = [1, 2, 5]  # 扫描 3 个距离 (Python float)

# 存储结果
center_intensities = []
all_wavefronts = []
grid_dir = rave_sim_dir / ""
grid_paths = [
    str(grid_dir / f"thin_w_plate_{t}um_200um.npy")
    for t in w_width
]
print(grid_paths)
for gp in enumerate(grid_paths):    
    config_dict = {
        "sim_params": {
            "is2d": 'true',
            "N": 16384 * 16384,
            "nx": 16384,
            "ny": 16384,
            "dx": 6.0e-8,          # 65 nm (z=0.5m 时的频率约束: dx < λ·z / detector_size)
            "z_detector": z_det,   # 扫描参数
            "detector_size": 0.85e-3,
            "detector_size_y": 0.85e-3,
            "detector_pixel_size_x": 2e-7,
            "detector_pixel_size_y": 2e-7,
            "chunk_size": 2*1024*1024*1024 // 16,
        },
        "use_disk_vector": False,
        "save_final_u_vectors": False,
        "dtype": "c8",
        "multisource": {
            "type": "points",
            "energy_range": [9999, 10001],
            "x_range": [-1e-6, 1e-6],
            "y_range": [-1e-6, 1e-6],
            "z": 0.0,
            "nr_source_points": 1,
            "seed": 1,
        },
        "elements": [
            "type": "sample",
            "z_start": 1.0,            # 样品位于 1.0 m
            "pixel_size_x": 5e-8,      # 样品内部高分辨率
            "pixel_size_y": 5e-8,
            "pixel_size_z": 5e-8,
            "grid_path":gp,
            "materials": [["W", 19.35]],
            "x_positions": [0],
            "y_positions": [0],
        ],
    }
    
    # 设置仿真
    sim_path = multisim.setup_simulation(config_dict, Path("."), simulations_dir)
    computed = config.load(Path(sim_path / 'computed.yaml'))
    
    # 运行仿真
    for i in tqdm(range(config_dict["multisource"]["nr_source_points"])):
        os.system(f"CUDA_VISIBLE_DEVICES=0 /mnt/d/rave-sim-main/rave-sim-main/fast-wave/build-Release/fastwave -s {i} {sim_path}")
    
    # 加载波场
    wavefronts = util.load_wavefronts_filtered(sim_path, x_range=(-30e-6, 30e-6))
    wavef = [result[0] for result in wavefronts]
    wf = np.sum(wavef, axis=0)
    
    # 提取中心强度（中心 100x100 像素平均）
    det = wf[0].astype(np.float64)
    ny, nx = det.shape
    cy, cx = ny // 2, nx // 2
    center_mean = det[cy-50:cy+50, cx-50:cx+50].mean()
    center_intensities.append(center_mean)
    all_wavefronts.append(det)
    
    print(f"  Center intensity: {center_mean:.4e}")
    
    # 保存该距离的波场到文件
    from contextlib import redirect_stdout
    with open(f'output_distance_z_{z_det:.1f}m.txt', 'w') as file:
        with redirect_stdout(file):
            for row in range(len(det)):
                print(det[row])
    print(f"  Saved output_distance_z_{z_det:.1f}m.txt")

print(f"\n{'='*60}")
print("Scan complete!")
print(f"z_distances: {z_distances}")
print(f"center_intensities: {center_intensities}")
print('='*60)

SyntaxError: invalid syntax (3466813995.py, line 42)